In [55]:
import csv
import json
import logging

logging.basicConfig(level=logging.INFO, format='%(levelname)s:%(message)s')

def make_report(csv_path: str, json_path: str) -> int:
    results = []
    processed_count = 0

    try:
        with open(csv_path, "r", encoding="utf-8") as f:
            reader = csv.DictReader(f)
            for row in reader:
                name = row['이름'].strip()
                student_id = row['학번'].strip()

                mid = row['중간'].strip()
                fin = row['기말'].strip()
                ass = row['과제'].strip()

                # 빈 칸 확인 → 하나라도 비면 평균/등급 None
                if not mid or not fin or not ass:
                    scores_dict = {
                        "중간": int(float(mid)) if mid else None,
                        "기말": int(float(fin)) if fin else None,
                        "과제": int(float(ass)) if ass else None
                    }
                    avg = None
                    grade = None
                    logging.info(f"{name}: 평균 None, 등급 None (결측값 포함)")
                else:
                    m, f_score, a = float(mid), float(fin), float(ass)
                    avg = m * 0.3 + f_score * 0.5 + a * 0.2

                    if avg >= 90:   grade = "A"
                    elif avg >= 80: grade = "B"
                    elif avg >= 70: grade = "C"
                    else:           grade = "F"

                    scores_dict = {"중간": int(m), "기말": int(f_score), "과제": int(a)}
                    logging.info(f"{name}: 평균 {avg}, 등급 {grade}")

                results.append({
                    "이름": name,
                    "학번": student_id,
                    "점수": scores_dict,
                    "평균": avg,
                    "등급": grade
                })
                processed_count += 1

        with open(json_path, "w", encoding="utf-8") as jf:
            json.dump(results, jf, ensure_ascii=False, indent=4)

        return processed_count

    except FileNotFoundError:
        logging.warning(f"파일이 없습니다: {csv_path}")
        return 0
    except UnicodeDecodeError:
        logging.error(f"인코딩 오류가 발생했습니다: {csv_path}")
        return 0

make_report("scores.csv", "report.json")

INFO:root:김언어: 평균 89.5, 등급 B
INFO:root:이국문: 평균 84.4, 등급 B
INFO:root:박영문: 평균 93.5, 등급 A
INFO:root:최역사: 평균 None, 등급 None (결측값 포함)


4

In [48]:
import logging


class InvalidJamoError(ValueError):
    """한글 자음이나 모음이 아닌 문자가 들어왔을 때 발생하는 예외"""
    pass


def classify_jamo(c: str) -> str:
    
    if not isinstance(c, str):
        raise TypeError(f"문자열이 아닙니다: {type(c).__name__}")
    
    
    if len(c) != 1:
        raise ValueError(f"한 글자만 입력해야 합니다: '{c}'")
    
    code = ord(c)
    
    
    if 0x3131 <= code <= 0x314E:
        return "자음"
    
    elif 0x314F <= code <= 0x3163:
        return "모음"

    else:
        raise InvalidJamoError(f"유효한 한글 자모가 아닙니다: '{c}'")

inputs = ["ㄱ", "ㅏ", "ㅎ", "ㅣ", "AB", 5, "!", " ", ""]

for item in inputs:
    try:
        result = classify_jamo(item)
        print(result)
    except TypeError as e:
        print(f"[TypeError] {e}")
    except InvalidJamoError as e:
        print(f"[InvalidJamoError] {e}")
    except ValueError as e:
        print(f"[ValueError] {e}")

자음
모음
자음
모음
[ValueError] 한 글자만 입력해야 합니다: 'AB'
[TypeError] 문자열이 아닙니다: int
[InvalidJamoError] 유효한 한글 자모가 아닙니다: '!'
[InvalidJamoError] 유효한 한글 자모가 아닙니다: ' '
[ValueError] 한 글자만 입력해야 합니다: ''
